In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = resnet18(
    weights = ResNet18_Weights.DEFAULT
)

model = model.to(device)

model.eval()

x = torch.randn(4,3,224,224).to(device)

with torch.inference_mode():
  output = model(x)

print(output.shape)
print(output)

torch.Size([4, 1000])
tensor([[ 0.6445,  2.4046,  2.9820,  ...,  0.3214,  0.0730,  1.1148],
        [ 0.9942,  2.7658,  2.6934,  ...,  0.1043,  0.3762,  1.3142],
        [ 0.6916,  2.2570,  2.7580,  ..., -0.3562,  0.1906,  1.5131],
        [ 0.3680,  2.7304,  2.4164,  ...,  0.2544,  0.7458,  1.4941]])


In [ ]:
import torch
import torch.nn as nn

nureon1 = nn.Identity()
nureon2 = nn.ReLU()
x = torch.tensor([1.0, 2.0, 3.0])
y = nureon1(x) + nureon2(x)

print(y)

tensor([2., 4., 6.])


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Set the random seed
torch.manual_seed(42)

# Set the device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create the random tensor and move it to device
x = torch.randn(4,3,224,224).to(device)


## Set up model1
model1 = resnet18(
    weights = ResNet18_Weights.DEFAULT
)

model1.layer2[1] = nn.Identity()

model1 = model1.to(device)

model1.eval()

with torch.inference_mode():
  output1 = model1(x)

## Set up model2
model2 = resnet18(
    weights = ResNet18_Weights.DEFAULT
)

model2 = model2.to(device)

model2.eval()

with torch.inference_mode():
  output2 = model2(x)

print(output1)
print(output2)

tensor([[ 1.9304,  1.8479, -2.3283,  ...,  6.5416,  0.9284,  4.1784],
        [ 2.5113,  2.1736, -1.3625,  ...,  6.5610,  0.3821,  3.8737],
        [ 2.4587,  2.2892, -2.0397,  ...,  7.0468,  0.3011,  3.7425],
        [ 2.1540,  2.0113, -1.7571,  ...,  6.6294,  0.3169,  4.0146]])
tensor([[ 0.2887,  2.5386,  2.2874,  ...,  0.2134,  0.3308,  1.4238],
        [ 0.5757,  2.6891,  3.0858,  ..., -0.0168,  0.0641,  1.3329],
        [ 0.3168,  2.5959,  2.6198,  ..., -0.0055, -0.0207,  1.4384],
        [ 0.4354,  2.6377,  2.8349,  ...,  0.1687,  0.4899,  1.3223]])


A simple ablation experiment

In [ ]:

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Set the random seed
torch.manual_seed(42)

# Set the device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create the random tensor and move it to device
x = torch.randn(4,3,224,224).to(device)

# Define three activation functions
activations = {
    "ReLU": nn.ReLU,
    "LeakyReLU": nn.LeakyReLU,
    "GELU": nn.GELU
}

# Create result dictionary
results = {}

with torch.inference_mode():

  for name, activation in activations.items():

    model = resnet18(
        weights = ResNet18_Weights.DEFAULT
    )

    model.layer2[1].relu = activation()

    model = model.to(device)

    model.eval()

    output = model(x)

    results[name] = output.cpu()
    print(f"{name}:{output}")


print(nn.ReLU)

ReLU:tensor([[ 0.2887,  2.5386,  2.2874,  ...,  0.2134,  0.3308,  1.4238],
        [ 0.5757,  2.6891,  3.0858,  ..., -0.0168,  0.0641,  1.3329],
        [ 0.3168,  2.5959,  2.6198,  ..., -0.0055, -0.0207,  1.4384],
        [ 0.4354,  2.6377,  2.8349,  ...,  0.1687,  0.4899,  1.3223]])
LeakyReLU:tensor([[ 0.1662,  2.5329,  2.3619,  ...,  0.0959,  0.2664,  1.4852],
        [ 0.5116,  2.7226,  3.1614,  ..., -0.0906,  0.0656,  1.3722],
        [ 0.2083,  2.6408,  2.7275,  ..., -0.0649, -0.0254,  1.4367],
        [ 0.3674,  2.6528,  2.9279,  ...,  0.0612,  0.4498,  1.3471]])
GELU:tensor([[ 0.1073, -0.4280, -1.0439,  ..., -2.3621, -0.2819,  2.6923],
        [ 0.2215, -0.3692, -0.7785,  ..., -2.4115, -0.2352,  2.5802],
        [ 0.2139, -0.3691, -1.1766,  ..., -2.7188, -0.3763,  2.5625],
        [ 0.2542, -0.3143, -1.1742,  ..., -2.5711, -0.3256,  2.5837]])
<class 'torch.nn.modules.activation.ReLU'>


We have the following three common method to deal with catastrophic forgetting:

 - Regularization
 - Replay
 - Parameter Isolation

# Do fine-tune on a pretrained Resnet

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import dataloader, random_split



## Set up random seeds and device

- random seeds determine every things related to random number generation
- when setting a new model you must consider the setting of random seed

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42
EPOCHS = 10
BATCH_SIZE = 128

def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
